# PathSnap Member 3: Model Training and Bayesian Optimisation

This notebook demonstrates candidate-road binary classification using Logistic Regression, KNN, Decision Tree, Random Forest, and HistGradientBoosting.

**Important:** Real candidate features and labels are unavailable. Synthetic scores in this notebook are smoke-test results only, not PathSnap benchmarks.

## 1. Project Context and Objectives

Member 3 estimates whether a candidate road segment is correct (`1`) or incorrect (`0`). This notebook covers synthetic loading, feature selection, trip-aware GroupKFold validation, model comparison, Optuna TPE optimisation, model saving, and integration-ready probability output.

## 2. Load Candidate-Level Feature Data

Real candidate-level features are not in the workspace, so this section uses reproducible synthetic trip data only.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

X, y = make_classification(n_samples=240, n_features=8, n_informative=5, n_redundant=1, random_state=42)
candidate_data = pd.DataFrame(X, columns=[f"feature_{i}" for i in range(X.shape[1])])
candidate_data["label"] = y
candidate_data["trip_id"] = np.repeat([f"trip_{i:03d}" for i in range(12)], 20)
candidate_data["gps_id"] = np.arange(len(candidate_data))
candidate_data["candidate_id"] = np.arange(len(candidate_data))
candidate_data["arc_id"] = np.arange(len(candidate_data)) % 25
candidate_data.loc[candidate_data.index % 31 == 0, "feature_0"] = np.nan

print("Required columns present:", {"trip_id", "label"}.issubset(candidate_data.columns))
candidate_data.head()

## 3. Feature Selection and Preprocessing

Predictive features exclude trip, GPS, candidate, arc, and label identifiers. Each sklearn pipeline performs fold-local median imputation and scaling.

In [ ]:
feature_columns = [f"feature_{i}" for i in range(8)]
print("Predictive feature columns:", feature_columns)
print("Missing values:", int(candidate_data[feature_columns].isna().sum().sum()))
print("Infinite values:", int(np.isinf(candidate_data[feature_columns].to_numpy()).sum()))

## 4. Label Distribution Analysis

In [ ]:
label_counts = candidate_data["label"].value_counts().sort_index()
print(label_counts)
ax = label_counts.plot(kind="bar", title="Synthetic Candidate Label Distribution")
ax.set_xlabel("Candidate label")
ax.set_ylabel("Candidate count")
ax.set_xticklabels(["0: incorrect", "1: correct"], rotation=0)
plt.tight_layout()
plt.show()

## 5. Group-Aware Cross-Validation Setup

GroupKFold uses `trip_id` so no trip occurs in both a training and validation fold.

In [ ]:
from src.evaluation.cross_validation import get_group_splits

splits = get_group_splits(candidate_data, trip_id_column="trip_id", n_splits=5)
print("Unique trips:", candidate_data["trip_id"].nunique())
print("Effective folds:", len(splits))
for fold_number, (train_indices, validation_indices) in enumerate(splits, start=1):
    train_trips = set(candidate_data.iloc[train_indices]["trip_id"])
    validation_trips = set(candidate_data.iloc[validation_indices]["trip_id"])
    assert train_trips.isdisjoint(validation_trips)
    print(f"Fold {fold_number}: train trips={len(train_trips)}, validation trips={len(validation_trips)}, overlap=0")

## 6. Train and Compare Five Models

In [ ]:
from src.models.training import train_and_evaluate_models
from src.evaluation.model_comparison import build_model_comparison

synthetic_results = train_and_evaluate_models(
    candidate_data,
    feature_columns=feature_columns,
    label_column="label",
    trip_id_column="trip_id",
    n_splits=5,
    random_state=42,
    output_directory="experiments/results",
    synthetic_data_only=True,
)
comparison = build_model_comparison(synthetic_results)
comparison

## 7. Evaluate Model Metrics

Primary model-selection metric: mean F1. Accuracy, precision, recall, ROC-AUC, and confusion matrices are secondary evidence.

In [ ]:
for model_name, model_result in synthetic_results["models"].items():
    print(f"\n{model_name}")
    display(model_result["aggregate"])
    display(pd.DataFrame(model_result["fold_results"])[["fold", "accuracy", "precision", "recall", "f1", "roc_auc", "training_time_seconds", "inference_time_seconds"]])

## 8. Visualise Model Comparison Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
comparison.sort_values("mean_f1").plot(x="model", y="mean_f1", kind="barh", ax=axes[0], legend=False, title="Synthetic Mean F1")
comparison.sort_values("mean_training_seconds").plot(x="model", y="mean_training_seconds", kind="barh", ax=axes[1], legend=False, title="Synthetic Mean Training Time")
plt.tight_layout()
plt.show()
print("Synthetic mean-F1 leader:", comparison.iloc[0]["model"])
comparison

## 9. Bayesian Hyperparameter Optimisation

This is a small Optuna TPE smoke run with group-aware folds. The best tuning objective is not an unbiased generalisation estimate.

In [ ]:
from src.optimization.bayesian_optimization import optimize_model

baseline_model_name = comparison.iloc[0]["model"]
optimization = optimize_model(
    candidate_data,
    baseline_model_name,
    feature_columns=feature_columns,
    n_trials=2,
    n_splits=5,
    random_seed=42,
    output_directory="experiments/results",
)
print("Optimised model:", optimization["model_name"])
print("Synthetic best objective:", optimization["best_objective_value"])
print("Best hyperparameters:", optimization["best_hyperparameters"])

## 10. Baseline vs Optimised Model Comparison

Baseline and optimised scores are compared only in synthetic smoke tests here. Real held-out data is required before claiming improvement.

In [ ]:
baseline_row = comparison.loc[comparison["model"] == baseline_model_name].iloc[0]
print("Baseline synthetic mean F1:", baseline_row["mean_f1"])
print("Optimised synthetic tuning objective:", optimization["best_objective_value"])
print("A held-out real-data evaluation is required before claiming improvement.")

## 11. Model Saving and Artifact Generation

This synthetic smoke artifact includes feature order and training metadata.

In [ ]:
from src.models.model_io import save_model

model_path, metadata_path = save_model(
    optimization["model"],
    {
        "hyperparameters": optimization["best_hyperparameters"],
        "feature_columns": feature_columns,
        "random_seed": 42,
        "training_config": {"cross_validation": "GroupKFold", "n_splits": 5, "synthetic_smoke_test": True},
        "validation_metrics": {"synthetic_objective": optimization["best_objective_value"]},
        "feature_schema_version": "pending_member_2_schema",
    },
    output_directory="artifacts",
    model_name=baseline_model_name,
)
print("Model artifact:", model_path)
print("Metadata artifact:", metadata_path)

## 12. Integration Output Demonstration

The output preserves trip, GPS, candidate, and arc identifiers. Member 2 can use `positive_probability` as candidate emission input for HMM/Viterbi.

In [ ]:
from src.models.prediction import predict_candidates

predictions = predict_candidates(candidate_data.head(20), model_path)
predictions.head()